# 01 - Exploratory Data Analysis (EDA) - Sales Time Series

Este notebook importa nuestro `SalesTimeSerieExtractor` oficial de `ml/src/data/make_dataset.py` para visualizar y explorar las señales estacionales o tendencias antes de empujar el RandomForest a producción.

In [ ]:
# Añadir la ruta raíz al sys.path para importaciones de módulos ml/src
import sys
import os
sys.path.append(os.path.abspath('..'))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.make_dataset import SalesTimeSerieExtractor

sns.set_theme(style="darkgrid")
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)

### 1. Conexión y Carga de Datos EDW

In [ ]:
# Inicializamos el extractor a la base de PostgreSQL configurado en el .env
extractor = SalesTimeSerieExtractor()

# Traemos toda la historia global de ventas
try:
    df_raw = extractor.fetch_daily_sales()
    display(df_raw.head())
except Exception as e:
    print("Asegúrate de que PostgreSQL está activo y las variables de entorno están cargadas. ", e)

### 2. Visualización de Tendencia Histórica

In [ ]:
if 'df_raw' in locals() and not df_raw.empty:
    plt.plot(df_raw.index, df_raw['y_sales_net'], color='blue', alpha=0.6, label='Ventas Diarias')
    
    # Agregar un Rolling Mean de 7 días (Tendencia Suavizada)
    plt.plot(df_raw['y_sales_net'].rolling(7).mean(), color='red', linewidth=2, label='Media Móvil 7d')
    
    plt.title('Tendencia de Ventas (Time Series)')
    plt.ylabel('Ventas Netas ($)')
    plt.xlabel('Fecha')
    plt.legend()
    plt.show()

### 3. Prueba de Pipeline de Features (Lags)
Verificamos si nuestro Pipeline `build_features` funciona como se espera, generando ventas retrasadas (lags).

In [ ]:
from src.features.build_features import build_preprocessing_pipeline

if 'df_raw' in locals() and not df_raw.empty:
    pipeline = build_preprocessing_pipeline()
    df_features = pipeline.fit_transform(df_raw)
    display(df_features.head(10))